# TD 3 — Extracting information with an LLM 🤖

## From travel stories to a map 📍

### Objective 🎯

You have six travel stories in French. Each one tells a week in a department of the Provence-Alpes-Côte d'Azur region. You will:

1. **extract** the places with an LLM;
2. **geocode** these places with the Géoplateforme API;
3. **draw** the map.

The geocoder contains several indexes, and the choice matters a lot:

| Index | Content | Example |
| --- | --- | --- |
| `address` | the addresses of the French national address database (BAN): municipalities, streets, numbers | Digne-les-Bains |
| `poi` | points of interest: peaks, lakes, museums, neighbourhoods... | lac d'Allos |

### Rules of the game 📏

- Information missing from the text is `null`. The model must **never** invent coordinates.
- Always keep the name of the source file.
- A valid JSON is not always a **correct** JSON: check.

## Set up the environment ⚙️

This lab works with a **local** model (Ollama, see TD 2) or a **hosted** one (Gemini, see TD 1). Choose in the next cell. Packages needed: `pydantic requests folium`, + `ollama` / `google-genai python-dotenv`.

The `ask` function sends a prompt and returns the text of the answer. If you give it a Pydantic schema, the answer is JSON.

> A local model with 3 billion parameters makes mistakes: this is normal, and interesting to observe. Do not use FunctionGemma here, it is not made for reading texts.

In [ ]:
import os
import json
from pathlib import Path
from time import sleep

BACKEND = "ollama"            # "ollama" (local) or "gemini" (hosted)
MODELS = {"ollama": "qwen2.5:3b", "gemini": "gemini-3.5-flash-lite"}

def ask(prompt, schema=None, model=None, backend=None):
    """Send a prompt, return the text of the answer (JSON if a schema is given)."""
    backend = backend or BACKEND
    model = model or MODELS[backend]

    if backend == "ollama":
        import ollama
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            options={"temperature": 0, "num_ctx": 8192, "num_predict": 3000},
        )
        return response.message.content

    from dotenv import load_dotenv
    from google import genai
    from google.genai import errors, types
    load_dotenv()
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    config = types.GenerateContentConfig(
        response_mime_type="application/json" if schema else None,
        response_schema=schema,
    )
    for attempt in range(4):
        try:
            return client.models.generate_content(model=model, contents=prompt, config=config).text
        except errors.APIError as error:
            if error.code not in (429, 500, 503) or attempt == 3:
                raise
            sleep(2 ** attempt)

print(ask("Say hello in French, in three words."))

## Part 1 — Extract the places 📍

❓ **>>> Exercise 1.** Read the six `.txt` files of the `data/texts/` folder into a dictionary `stories`, whose keys are the file names. Display the number of words in each story.

Here is the structure to obtain for **one** story:

```json
{
  "source_file": "voyage_alpes_de_haute_provence.txt",
  "places": [
    {
      "name": "lac d'Allos",
      "type": "lake",
      "department": "Alpes-de-Haute-Provence",
      "municipality": null,
      "visit_status": "visited"
    }
  ]
}
```

`visit_status` can be:

| Value | Meaning |
| --- | --- |
| `visited` | the traveller went there |
| `seen` | the traveller only saw it, from a distance |
| `considered` | the traveller thought about going, but did not |
| `mentioned` | the place is named, but neither seen nor considered |
| `unknown` | the text does not say |

In the example, the municipality is `null`: the story does not say in which municipality the lake is.

❓ **>>> Exercise 2.** Write two Pydantic schemas:

- `Place`, with the five fields above (use `Literal` for `visit_status`);
- `Places`, which contains a list `places`.

Test that `Place` **refuses** an unknown `visit_status`.

❓ **>>> Exercise 3.** Write a function `extract_places(text)` that:

1. builds a precise prompt (context, definitions, rules, text);
2. calls `ask` with your schema;
3. validates the answer and returns the list of places;
4. tries **once** more if the answer is invalid.

Test it on a single story.

❓ **>>> Exercise 4.** The same place may appear twice ("Marseille", "marseille", "le Vieux-Port", "Vieux-Port").

1. Write a function `normalize(name)`: lower case, no accents, no article at the beginning, hyphens replaced by spaces.
2. Write a function `remove_duplicates(places)` that only keeps the first mention.

❓ **>>> Exercise 5.** Process the six stories and save the result in `output/places.json`: a list of dictionaries `{"source_file": ..., "places": [...]}`. Accents must remain readable in the file.

❓ **>>> Exercise 6.** Look for errors **automatically**: display the extracted places whose name does not appear in the story. They may be inventions, or modified names.

Then read **one** story yourself: which places are missing? Which `visit_status` values are wrong?

## Part 2 — Geocode 🌍

The Géoplateforme [geocoding service](https://cartes.gouv.fr/aide/fr/guides-utilisateur/utiliser-les-services-de-la-geoplateforme/geocodage/) is free and needs no key. Its limits are generous (50 requests per second), but do not send the same request twice.

Useful parameters: `q` (text to search), `index` (`address` or `poi`), `limit`, `depcode` (department code). Let's compare both indexes:

In [ ]:
import requests

GEOCODER = "https://data.geopf.fr/geocodage/search"

def search(query, index="address", limit=1, depcode=None):
    """Return the list of candidates found by the geocoder."""
    parameters = {"q": query, "index": index, "limit": limit}
    if depcode:
        parameters["depcode"] = depcode
    response = requests.get(GEOCODER, params=parameters, timeout=20)
    response.raise_for_status()
    return response.json()["features"]

for query in ("Digne-les-Bains", "lac d'Allos"):
    for index in ("address", "poi"):
        for feature in search(query, index):
            p = feature["properties"]
            label = p.get("label") or p.get("toponym")
            print(f'{query:16s} {index:8s} score={p["score"]:.2f}  {label}  {feature["geometry"]["coordinates"]}')

> Watch out: the order is **[longitude, latitude]**.

> The `score` measures the similarity **of the text**. It does not say whether the position is accurate, nor even whether it is the right place.

❓ **>>> Exercise 7.** Search for "Notre-Dame de la Garde":

1. in both indexes, **without** a department;
2. in both indexes, with `depcode="13"`.

Display the label, the city, the department and the score. What do you notice?

Here are two examples of results. An accepted match:

```json
"geocoding": {
  "status": "accepted",
  "matched_label": "Lac d'Allos",
  "city": "Allos",
  "category": "lac",
  "score": 0.85,
  "longitude": 6.708566,
  "latitude": 44.233245
}
```

A doubtful match: for "place des Lices" (in Saint-Tropez), the geocoder suggested a street in Fréjus in November 2026.

```json
"geocoding": {
  "status": "uncertain",
  "matched_label": "Rue des LICES 83600 Fréjus",
  "city": "Fréjus",
  "category": "street",
  "score": 0.57,
  "longitude": null,
  "latitude": null
}
```

A place that cannot be found, such as "sentier des Ocres" (Vaucluse): no suitable candidate.

```json
"geocoding": {
  "status": "not_found",
  "matched_label": null,
  "city": null,
  "category": null,
  "score": null,
  "longitude": null,
  "latitude": null
}
```

| Status | Coordinates |
| --- | --- |
| `accepted` | filled in |
| `uncertain` | `null`; the label and the score of the candidate are kept for a human check |
| `not_found` | `null` |

Never replace a specific site with the centre of its municipality without saying so.

In [ ]:
# Department code of each story
DEPCODES = {
    "voyage_alpes_de_haute_provence.txt": "04",
    "voyage_hautes_alpes.txt": "05",
    "voyage_alpes_maritimes.txt": "06",
    "voyage_bouches_du_rhone.txt": "13",
    "voyage_var.txt": "83",
    "voyage_vaucluse.txt": "84",
}

❓ **>>> Exercise 8.** Write a function `geocode(name, depcode)` that returns a `geocoding` dictionary:

1. search the `poi` index, then the `address` index, with the department;
2. keep the candidate with the highest score;
3. choose a status from the score **and** from the similarity of the names (use `normalize`);
4. if the API fails, the program must not stop: use the status `error`;
5. use `functools.lru_cache` so as not to repeat a request.

Test it with "lac d'Allos" (04), "place des Lices" (83), "sentier des Ocres" (84) and "Vieux-Port" (13). Look closely at the **city** of the last one.

❓ **>>> Exercise 9.** Geocode all the places of `output/places.json`. Add a `geocoding` object to each place, without losing the original information, and save to `output/places_geocoded.json`. Count the statuses.

## Part 3 — The map 🗺️

❓ **>>> Exercise 10.** With `folium`, draw the accepted places. Use one colour per `visit_status`, and display the name, the matched label and the category in the popup. Save the map to `output/map.html`.

## Discussion 💭

- Which places were easy or difficult to extract? Did the model confuse `seen` or `considered` with `visited`?
- Look for other matches that are **accepted but wrong**. What information would have avoided them?
- Do the coordinates represent the specific site, a street or a wider area? Compare a few of them with a reliable map.

## Deliverables 📦

- This notebook, with the extraction, the geocoding and the map.
- `places.json` and `places_geocoded.json`.
- A short discussion of errors and limitations. Do not include your API key.

## Going further (bonus) 🚀

- Turn your code into a script that can be used from the terminal, with paths as parameters.
- Put the application in a Docker container.